In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import torch.nn.functional as F
import numpy as np

# === Load datasets ===
imdb = load_dataset("imdb")
s140 = {
    "train": load_dataset("contemmcm/sentiment140", split="complete[:50000]"),
    "test": load_dataset("contemmcm/sentiment140", split="complete[50000:60000]")
}
# === Preprocessing ===
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === Tokenization & Vocab ===
def tokenize(text):
    return clean_text(text).split()

# Build vocab from IMDB
counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

# Only keep words with min_freq ≥ 5
vocab = {"<pad>": 0, "<unk>": 1}
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)
    ids = [vocab.get(token, UNK_IDX) for token in tokens[:max_len]]
    return torch.tensor(ids, dtype=torch.long)

# === Dataset Wrappers ===
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text_field = item["text"] if "text" in item else item["content"]
        tokens = text_pipeline(text_field, max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)



/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


README.md:   0%|          | 0.00/361 [00:00<?, ?B/s]

polarity.csv:   0%|          | 0.00/124M [00:00<?, ?B/s]

Generating complete split: 0 examples [00:00, ? examples/s]

In [ ]:
train_loader_imdb = DataLoader(TextDataset(imdb["train"], lambda x: x), batch_size=64, shuffle=True, collate_fn=collate_batch)
train_loader_s140 = DataLoader(TextDataset(s140["train"], lambda x: x), batch_size=256, shuffle=True, collate_fn=collate_batch)
test_loader_s140 = DataLoader(TextDataset(s140["test"], lambda x: x), batch_size=256, shuffle=False, collate_fn=collate_batch)

# === Model ===
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextClassifier(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [2]:
# === Sliced Wasserstein ===
from sliced_wasserstein import sliced_wasserstein_distance

def feature_extractor(loader, max_samples=1000):
    features = []
    model.eval()
    sample_count = 0
    
    with torch.no_grad():
        for x, _ in loader:
            if sample_count >= max_samples:
                break
            x = x.to(device)
            feature = model(x)
            features.append(feature.cpu().numpy())
            sample_count += x.shape[0]
        return np.concatenate(features, axis=0)

imdb_features = feature_extractor(train_loader_imdb)
print("Extracted imdb features")

s140_features = feature_extractor(train_loader_s140)
print("Extracted sentiment140 features")

print("Sliced Wasserstein Distance between sentiment140 and imdb:", 
      sliced_wasserstein_distance(s140_features, imdb_features))


# === Jensen-Shannon Divergence ===
from sklearn.feature_extraction.text import CountVectorizer

def KL_numpy(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    epsilon = 1e-10
    b = np.where(b == 0, epsilon, b)
    return np.sum(np.where(a != 0, a * np.log(a / b), 0))

imdb_small = load_dataset("imdb", split='train[:10000]')
s140_small = load_dataset("contemmcm/sentiment140", split='complete[60000:70000]')

texts_imdb = [x['text'] for x in imdb_small]
texts_s140 = [x['text'] for x in s140_small]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_s140)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_s140 = vectorizer.transform(texts_s140).toarray().astype(np.float32)

def to_prob_dist(data):
    data = data + 1e-10
    return data / data.sum(axis=1, keepdims=True)

datasets = {
    'IMDB': X_imdb,
    'S140': X_s140
}

prob_datasets = {}
avg_datasets = {}

for name, data in datasets.items():
    prob_datasets[name] = to_prob_dist(data)
    avg_datasets[name] = prob_datasets[name].mean(axis=0)

print("Jensen-Shannon Divergence between dataset pairs:")
for name1 in datasets.keys():
    for name2 in datasets.keys():
        if name1 != name2:
            m = (avg_datasets[name1] + avg_datasets[name2]) * 0.5
            js = 0.5 * KL_numpy(avg_datasets[name1], m) + 0.5 * KL_numpy(avg_datasets[name2], m)
            print(f"{name1} vs {name2}: {js:.12f}")


# === Gradient Cosine + L2 Norm ===
def get_gradient_vector(model, dataloader):
    model.train()
    model.zero_grad()
    x, y = next(iter(dataloader))
    x, y = x.to(device), y.to(device)

    out = model(x)
    loss = F.cross_entropy(out, y)
    loss.backward()

    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1))
    return torch.cat(grads)

grad_imdb = get_gradient_vector(model, train_loader_imdb)
grad_s140 = get_gradient_vector(model, train_loader_s140)

l2_diff = torch.norm(grad_imdb - grad_s140).item()
cos_sim = F.cosine_similarity(grad_imdb.unsqueeze(0), grad_s140.unsqueeze(0)).item()

print(f"L2 norm of gradient difference: {l2_diff:.12f}")
print(f"Cosine similarity between gradients: {cos_sim:.12f}")


# === MMD ===
from discrepancies import MaximumMeanDiscrepancy
from kernels import LaplacianKernel

imdb_small = load_dataset("imdb", split='train[:1000]')
s140_small = load_dataset("contemmcm/sentiment140", split='complete[70000:71000]')

texts_imdb = [x['text'] for x in imdb_small]
texts_s140 = [x['text'] for x in s140_small]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_s140)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_s140 = vectorizer.transform(texts_s140).toarray().astype(np.float32)

mmd = MaximumMeanDiscrepancy(kernel=LaplacianKernel(1e-2))
print("MMD between imdb and sentiment140:", mmd.compute(X_imdb, X_s140))


# === Hellinger Distance ===
def hellinger_np(p, q):
    return np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))

print("Hellinger Distance between imdb and sentiment140:", hellinger_np(X_imdb, X_s140))


Extracted imdb features
Extracted sentiment140 features
Sliced Wasserstein Distance between sentiment140 and imdb: 0.014025549249803848
Jensen-Shannon Divergence between dataset pairs:
IMDB vs S140: 0.222637729574
S140 vs IMDB: 0.222637729574
L2 norm of gradient difference: 0.186377689242
Cosine similarity between gradients: -0.646498441696
MMD between imdb and sentiment140: 0.27097747
Hellinger Distance between imdb and sentiment140: 223.2606
